In [1]:
!pip install --upgrade tdmclient

In [1]:
import tdmclient.notebook
await tdmclient.notebook.start()

# Motion Controller

In [2]:
import numpy as np
from tdmclient import ClientAsync

# ---------- Données ----------
init_position = [0, 0]
positions = [-5, 2, 10, 4]          # B = (-5, 2), C = (10, 4), en cm
current_position = list(init_position)
current_angle = 0.0                 # 0 = le robot regarde vers +y, sens horaire positif

# ---------- Paramètres à calibrer ----------
speed_turn = 100                    # vitesse des moteurs pour tourner
speed_forward = 150                 # vitesse des moteurs pour avancer
omega = 0.628                        # rad/s à speed_turn (À CALIBRER)
v_forward = 4.833                     # cm/s à speed_forward (À CALIBRER)
min_angle = np.radians(1)           # en dessous, on ne tourne pas

client = ClientAsync()


# ---------- Fonctions utiles ----------
def normalize(angle):
    """Ramène l'angle dans [-pi, pi] pour tourner par le plus court chemin."""
    return (angle + np.pi) % (2 * np.pi) - np.pi


def angle_to_target(pos, target):
    dx = target[0] - pos[0]
    dy = target[1] - pos[1]
    return np.arctan2(dx, dy)       # 0 = vers +y, positif = vers +x (horaire)


async def set_motors(node, left, right):
    await node.set_variables({
        "motor.left.target": [int(left)],
        "motor.right.target": [int(right)],
    })


# ---------- Rotation ----------
async def turn_to(node, target):
    global current_angle
    next_angle = angle_to_target(current_position, target)
    diff = normalize(next_angle - current_angle)

    if abs(diff) > min_angle:
        sign = 1 if diff > 0 else -1    # +1 = horaire (cible à droite)
        await set_motors(node, sign * speed_turn, -sign * speed_turn)
        await client.sleep(abs(diff) / omega)
        await set_motors(node, 0, 0)

    current_angle = next_angle


# ---------- Avancer ----------
async def go_forward(node, target):
    dx = target[0] - current_position[0]
    dy = target[1] - current_position[1]
    distance = np.hypot(dx, dy)

    await set_motors(node, speed_forward, speed_forward)
    await client.sleep(distance / v_forward)
    await set_motors(node, 0, 0)

    current_position[0], current_position[1] = target[0], target[1]


# ---------- Aller vers un point ----------
async def move_to(node, target):
    await turn_to(node, target)
    await go_forward(node, target)


# ---------- Programme principal ----------
async def main():
    node = await client.wait_for_node()
    await node.lock()

    B = positions[0:2]
    C = positions[2:4]

    await move_to(node, B)
    await move_to(node, C)

    await set_motors(node, 0, 0)
    await node.unlock()

client.run_async_program(main)

NodeLockError: Node lock error (current status: busy)

# Code pour calibrer la vitesse de rotation et de translation des roues

In [2]:
import numpy as np
from tdmclient import ClientAsync

speed_turn = 100
speed_forward = 150

duree = 5          # durée de chaque essai (s)
nb_essais = 3      # nombre d'essais
pause = 6          # temps entre deux essais (s)

client = ClientAsync()


async def set_motors(node, left, right):
    await node.set_variables({
        "motor.left.target": [int(left)],
        "motor.right.target": [int(right)],
    })


async def essais(node, left, right):
    for i in range(nb_essais):
        print(f"Essai {i + 1}/{nb_essais} dans 3 s... (replace le robot)")
        await client.sleep(3)
        print("GO")
        await set_motors(node, left, right)
        await client.sleep(duree)
        await set_motors(node, 0, 0)
        print("STOP : note le résultat.")
        if i < nb_essais - 1:
            await client.sleep(pause)


async def calibration(test):
    node = await client.wait_for_node()
    await node.lock()
    try:
        if test == "rotation":
            await essais(node, speed_turn, -speed_turn)
        else:
            await essais(node, speed_forward, speed_forward)
    finally:
        await set_motors(node, 0, 0)
        await node.unlock()

In [3]:
import tdmclient.notebook
await tdmclient.notebook.list()
#2 = available

id:       f203983e-5ac3-468c-8f9e-8d46dc4b0c91
group id: a26a8bf1-b940-40b7-9b21-d68946966af0
name:     Thymio 74
status:   4 (ready)
cap:      7
firmware: 14



In [4]:
await calibration("rotation")


Essai 1/3 dans 3 s... (replace le robot)
GO
STOP : note le résultat.
Essai 2/3 dans 3 s... (replace le robot)
GO
STOP : note le résultat.
Essai 3/3 dans 3 s... (replace le robot)
GO
STOP : note le résultat.


In [5]:
await calibration("avance")


Essai 1/3 dans 3 s... (replace le robot)
GO
STOP : note le résultat.
Essai 2/3 dans 3 s... (replace le robot)
GO
STOP : note le résultat.
Essai 3/3 dans 3 s... (replace le robot)
GO
STOP : note le résultat.


In [6]:
tours = [0.5, 0.5, 0.5]    # <- remplace par tes valeurs
omega = np.mean(tours) * 2 * np.pi / duree
print(f"omega = {omega:.3f} rad/s") 
print(f"écart : {np.std(tours) * 2 * np.pi / duree:.3f} rad/s")

omega = 0.628 rad/s
écart : 0.000 rad/s


In [8]:
distances = [24, 24.5, 24]    # <- tes mesures en cm
v_forward = np.mean(distances) / duree
print(f"v_forward = {v_forward:.3f} cm/s") 
print(f"écart : {np.std(distances) / duree:.3f} cm/s")

v_forward = 4.833 cm/s
écart : 0.047 cm/s
